# sortable_js

> Sortable.js CDN loading and initialization script generation.

In [ ]:
#| default_exp sortable_js

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fasthtml.common import Script

In [ ]:
#| export
SORTABLE_JS_CDN = "https://cdn.jsdelivr.net/npm/sortablejs@1.15.0/Sortable.min.js"

In [ ]:
#| export
def sortable_js_headers() -> tuple:  # Tuple of Script elements for app headers
    """CDN script tag for Sortable.js, suitable for FastHTML app headers."""
    return (Script(src=SORTABLE_JS_CDN),)

In [ ]:
#| export
def generate_sortable_init_script(
    handle_class: str = "drag-handle",  # CSS class for drag handle elements
    animation_ms: int = 150,  # Drag animation duration in milliseconds
) -> Script:  # Script element with Sortable.js initialization
    """Generate htmx.onLoad script that initializes Sortable.js on `.sortable` elements.

    Includes the disable-on-drag/re-enable-on-swap pattern to prevent
    double-firing during HTMX response processing.
    """
    return Script(f"""
htmx.onLoad(function(content) {{
    var sortables = content.querySelectorAll(".sortable");
    for (var i = 0; i < sortables.length; i++) {{
        var sortable = sortables[i];
        var sortableInstance = new Sortable(sortable, {{
            animation: {animation_ms},
            handle: ".{handle_class}",
            onEnd: function(evt) {{
                this.option("disabled", true);
            }}
        }});
        sortable.addEventListener("htmx:afterSwap", function() {{
            sortableInstance.option("disabled", false);
        }});
    }}
}});
""")

## Tests

In [ ]:
from fasthtml.common import to_xml

# CDN header
hdrs = sortable_js_headers()
assert len(hdrs) == 1
assert SORTABLE_JS_CDN in to_xml(hdrs[0])

# Init script — default params
script = generate_sortable_init_script()
xml = to_xml(script)
assert ".drag-handle" in xml
assert "animation: 150" in xml
assert "htmx.onLoad" in xml
assert 'this.option("disabled", true)' in xml
assert 'sortableInstance.option("disabled", false)' in xml

# Init script — custom params
script2 = generate_sortable_init_script(handle_class="my-handle", animation_ms=200)
xml2 = to_xml(script2)
assert ".my-handle" in xml2
assert "animation: 200" in xml2

print("All sortable_js tests passed")

All sortable_js tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()